# FABN — Clean vs Dirty Price / Yield Validation  *(fixed folder)*

This notebook **measures** the clean-vs-dirty yield bias on your real 303-bond book **and
verifies the fix** in this folder (the pipeline + backtest here solve the IRR against the **dirty**
price). It is the "Measure" + regression step of *Measure → fix → re-run*.

### The identity
$$\sum_t CF_t\,(1+y)^{-t} \;=\; P_{\text{dirty}} \;=\; P_{\text{clean}} + \text{AccruedInterest}$$
Quoted mid prices are **clean**. Solving against the clean price uses a target that is too small by
the accrued interest, so the solved yield is biased **UPWARD (overstated)**. Direction is derived
here and confirmed both synthetically (§0) and on the real book (§2).

**What changed in this folder vs the original `Optimization/`:**
- `fabn_finance.accrued_interest` / `previous_coupon_date` added (unit-tested).
- Pipeline §9.5 builds `dirty = price + accrued` and solves the IRR against `dirty`.
- Backtest Section 1 adds accrued per (day, bond) so `Y[d,i]` is a true YTM (kills the sawtooth).
- Clean `price` is unchanged as the carrying/book value and the bid-ask base.


## 0 — Self-contained sanity check (no BigQuery needed)
Proves the sign and magnitude on one hand-built bond. Runs anywhere.

In [1]:
import numpy as np
from scipy.optimize import brentq

def irr(cf, t, target_pv):
    cf = np.asarray(cf, float); t = np.asarray(t, float)
    return brentq(lambda y: float((cf*(1+y)**(-t)).sum() - target_pv), -0.5, 1.0, maxiter=200)

coupon_per_period, freq, periods, frac = 2.5, 2, 6, 0.5     # 5% semiannual, 3y, mid-period
t  = np.array([0.25 + 0.5*k for k in range(periods)])
cf = np.full(periods, coupon_per_period); cf[-1] += 100.0; cf = cf/100.0
clean_px = 100.0; accrued = coupon_per_period*frac; dirty_px = clean_px + accrued
y_clean = irr(cf, t, clean_px/100.0); y_dirty = irr(cf, t, dirty_px/100.0)
print(f"clean {clean_px:.2f} | accrued {accrued:.2f} | dirty {dirty_px:.2f}")
print(f"yield CLEAN (old) {y_clean*100:.4f}%  vs  DIRTY (correct) {y_dirty*100:.4f}%  "
      f"-> bias {(y_clean-y_dirty)*1e4:+.1f} bps")
assert y_clean > y_dirty
print("OK: clean-price omission biases the yield UPWARD.")

clean 100.00 | accrued 1.25 | dirty 101.25
yield CLEAN (old) 5.5680%  vs  DIRTY (correct) 5.0594%  -> bias +50.9 bps
OK: clean-price omission biases the yield UPWARD.


## 1 — Load the (fixed) pipeline
Bare `%run`, exactly like the working optimizer notebooks. Needs BigQuery/GCP ADC. If you lack
creds it errors loudly here (by design — the old silent `try/except` hid the real failure and
skipped §2–§5).

In [2]:
get_ipython().run_line_magic("run", "FABN_Data_Pipeline.ipynb")

Connected: insurance-backed-securities
Optimization date  : 2025-01-15
FABN issue/maturity: 2022-09-06 → 2027-09-06
Budget H           : $500,000,000
r_FABN             : 3.205%


/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Universe size N = 303 bonds


/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Spread coverage : 301/303 bonds  (2 missing)
Spread range    : -1.6 – 153.7 bps
Cashflow rows loaded : 2,517
Date range           : 2025-01-16 → 2033-02-15
bond_cf shape : (850, 303)  (T=850 payment dates × N=303 bonds)
qtr_bond_cf shape : (33, 303)  (Q=33 quarters × N=303 bonds)
Quarter range     : 2025Q1 → 2033Q1
Treasury curve date used : 2025-01-15  (FRED, attempt 1)
0.083yr     4.40
0.250yr     4.35
0.500yr     4.26
1.000yr     4.19
2.000yr     4.27
3.000yr     4.34
5.000yr     4.45
7.000yr     4.55
10.000yr    4.66
20.000yr    4.95
30.000yr    4.88

Duration computed from cashflows : 303
Duration from BBG fallback       : 0
Duration range                   : 1.16 – 6.40 yrs
Mean bond yield used             : 4.948%
theta range : 0.00158 – 0.02168
Mean C1     : 0.00911  (0.911%)
Rating source : 303 S&P, 0 Moody's fallback, 0 BBB default
h_curr (equal-weight) : $1,650,165.02 per bond
Full FABN schedule (per 100 face):
      date  coupon  principal    total
2023-03-06  1.6025       

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Accrued interest   : mean 1.128 / max 4.111 per 100 (added to clean price -> dirty-price YTM)
Mid price coverage : 301/303 bonds  (2 filled at par)
Book yield IRR     : 303/303 solved  (0 fell back to rf+spread)
Bid-ask tau        : 299/303 from quotes  (4 median-filled)
Book yield         : 4.33% – 6.01%  (mean 5.02%)
Coupon yield mean  : 4.28%   Amort yield mean : +0.736%
Bid-ask tau        : 1.1 – 29.7 bps  (mean 5.9 bps)


,Metric,Value,Notes
0,Universe size (N),303,
1,Payment dates (T),850,
2,Quarterly periods (Q),33,
3,Spread mean (bps),59.4,
4,Book yield mean (%),5.02,
5,Bid-ask tau mean (bps),5.9,
6,Duration mean (yrs),2.97,
7,C1 charge mean (%),0.911,
8,FABN D target (yrs),2.489641,
9,Budget H ($M),500.0,



pipeline dict ready.


In [3]:
import numpy as np, pandas as pd
import fabn_finance as ff
HAVE_PIPELINE = ("pipeline" in dir()) and isinstance(pipeline, dict)
print("pipeline loaded:", HAVE_PIPELINE,
      "| date:", optimization_date.date(), "| N:", len(pipeline["CUSIPS"]))

pipeline loaded: True | date: 2025-01-15 | N: 303


## 2 — Independent accrued interest, dirty yield, and **convention check**

We recompute everything *independently* (not reusing the pipeline's `accrued`) so this is a real
check, then ask: does `pipeline['book_yield']` match the **clean** or the **dirty** solve? In this
folder it must match **dirty** — that is the regression test that the fix is live.

In [4]:
if HAVE_PIPELINE:
    CUSIPS = list(pipeline["CUSIPS"]); N = len(CUSIPS)
    price   = np.asarray(pipeline["price"], float)       # clean mid, per 100
    bond_cf = np.asarray(pipeline["bond_cf"], float)     # (T,N) per $1
    t_vec   = np.asarray(pipeline["t_vec"], float)
    bk_pipe = np.asarray(pipeline["book_yield"], float)
    fixed_i = pipeline["fixed"].set_index("CUSIP")

    # independent accrued interest
    pay_dates = optimization_date + pd.to_timedelta(t_vec*365.25, unit="D")
    coupon100 = fixed_i.loc[CUSIPS, "coupon"].astype(float).values
    freq = fixed_i.loc[CUSIPS, "cpn_freq"].astype(float).values
    freq = np.where(np.isfinite(freq) & (freq > 0), freq, 2.0)
    has_cf = (bond_cf > 0).any(axis=0)
    first  = (bond_cf > 0).argmax(axis=0)
    next_cpn = pd.to_datetime(np.where(has_cf, pay_dates.values[first], np.datetime64("NaT")))
    prev_cpn = ff.previous_coupon_date(next_cpn, freq)
    accrued  = np.where(has_cf, ff.accrued_interest(optimization_date, prev_cpn, next_cpn,
                                                    coupon100, freq), 0.0)
    dirty = price + accrued

    y_clean = ff.book_yields(bond_cf, t_vec, price)      # what the OLD code did
    y_dirty = ff.book_yields(bond_cf, t_vec, dirty)      # the correct target
    ok = (~np.isnan(y_clean)) & (~np.isnan(y_dirty))

    d_clean = np.nanmax(np.abs(bk_pipe[ok]-y_clean[ok]))
    d_dirty = np.nanmax(np.abs(bk_pipe[ok]-y_dirty[ok]))
    conv = "DIRTY (fix is live)" if d_dirty < d_clean else "CLEAN (NOT fixed!)"
    print(f"Pipeline book_yield matches: {conv}")
    print(f"  max|pipe-clean|={d_clean:.2e}   max|pipe-dirty|={d_dirty:.2e}")

    bias = (y_clean[ok]-y_dirty[ok])*1e4
    print(f"\nAccrued (per 100): mean {accrued.mean():.3f}, max {accrued.max():.3f}, "
          f">0 for {(accrued>0).sum()}/{N} bonds")
    print(f"Yield bias clean-dirty (bps): mean {bias.mean():+.1f} | median {np.median(bias):+.1f} "
          f"| max {bias.max():+.1f} | >5bp on {(bias>5).sum()}/{ok.sum()} bonds")
else:
    print("skipped (no pipeline)")

Pipeline book_yield matches: DIRTY (fix is live)
  max|pipe-clean|=1.85e-02   max|pipe-dirty|=0.00e+00

Accrued (per 100): mean 1.128, max 4.111, >0 for 294/303 bonds
Yield bias clean-dirty (bps): mean +42.3 | median +36.5 | max +184.9 | >5bp on 258/303 bonds


In [5]:
if HAVE_PIPELINE:
    rpt = pd.DataFrame({
        "CUSIP": np.array(CUSIPS)[ok], "clean_px": np.round(price[ok],3),
        "accrued": np.round(accrued[ok],3), "dirty_px": np.round(dirty[ok],3),
        "y_clean_%": np.round(y_clean[ok]*100,4), "y_dirty_%": np.round(y_dirty[ok]*100,4),
        "bias_bps": np.round((y_clean[ok]-y_dirty[ok])*1e4,1),
    }).sort_values("bias_bps", ascending=False)
    print("Largest overstatements (what the OLD clean-price code added):")
    display(rpt.head(12))
else:
    print("skipped (no pipeline)")

Largest overstatements (what the OLD clean-price code added):


,CUSIP,clean_px,accrued,dirty_px,y_clean_%,y_dirty_%,bias_bps
40,61746BCY0,102.400,2.683,105.083,6.5618,4.7129,184.9
199,EF6184701,102.514,2.497,105.011,6.2307,4.5391,169.2
233,61238QAA6,105.485,2.685,108.170,6.8228,5.4744,134.8
293,205887AF9,103.768,2.075,105.843,6.1523,4.8544,129.8
95,78016HZT0,100.409,2.358,102.767,6.0225,4.7304,129.2
290,83368JKF6,98.232,1.721,99.953,6.7138,5.5079,120.6
248,78392BAE7,103.276,3.153,106.429,6.4277,5.2495,117.8
247,ZM2585630,103.276,3.153,106.429,6.4277,5.2495,117.8
156,TT3297450,104.431,1.927,106.358,6.1787,5.0025,117.6
242,880451AS8,104.363,2.320,106.683,6.0894,4.9297,116.0


## 3 — Portfolio-level dollar impact
How much annual statutory NII the **old** clean-price code overstated, on the current book.

In [6]:
if HAVE_PIPELINE:
    h = np.asarray(pipeline["h_curr"], float); H = float(pipeline.get("H", h.sum()))
    nii_clean = np.nansum(h[ok]*y_clean[ok]); nii_dirty = np.nansum(h[ok]*y_dirty[ok])
    print(f"Budget H                      : ${H:,.0f}")
    print(f"Annual NII @ clean (old)      : ${nii_clean:,.0f}")
    print(f"Annual NII @ dirty (fixed)    : ${nii_dirty:,.0f}")
    print(f"Overstatement removed by fix  : ${nii_clean-nii_dirty:,.0f}"
          f"  ({(nii_clean-nii_dirty)/max(nii_dirty,1)*100:+.2f}%)")
else:
    print("skipped (no pipeline)")

Budget H                      : $500,000,000
Annual NII @ clean (old)      : $27,212,134
Annual NII @ dirty (fixed)    : $25,096,560
Overstatement removed by fix  : $2,115,574  (+8.43%)


## 4 — Assertions

In [7]:
def check(name, cond, detail=""):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}" + (f"  -- {detail}" if detail else "")); assert cond, name

# (the synthetic sign was already asserted in §0)
if HAVE_PIPELINE:
    check("pipeline now uses DIRTY price (fix is live)", d_dirty < d_clean,
          f"clean {d_clean:.1e} vs dirty {d_dirty:.1e}")
    check("pipeline matches our dirty recompute (<1e-6)", d_dirty < 1e-6, f"{d_dirty:.2e}")
    finite = price[np.isfinite(price)]
    check("clean prices in [50,175]", finite.min()>=50 and finite.max()<=175,
          f"[{finite.min():.1f},{finite.max():.1f}]")
    check("0 <= accrued <= one period coupon", np.all(accrued>=-1e-9) and np.all(accrued<=coupon100/freq+1e-9))
    check("dirty >= clean", np.all(dirty>=price-1e-9))
    check("clean-price yield >= dirty-price yield (one-signed)", np.all(y_clean[ok]-y_dirty[ok]>=-1e-6))
    print("\nAll pipeline assertions passed.")
else:
    print("pipeline checks skipped.")

[PASS] pipeline now uses DIRTY price (fix is live)  -- clean 1.8e-02 vs dirty 0.0e+00
[PASS] pipeline matches our dirty recompute (<1e-6)  -- 0.00e+00
[PASS] clean prices in [50,175]  -- [80.7,120.7]
[PASS] 0 <= accrued <= one period coupon
[PASS] dirty >= clean
[PASS] clean-price yield >= dirty-price yield (one-signed)

All pipeline assertions passed.


## 5 — Verdict

- **Problem (confirmed on the real book):** the original `Optimization/` code solved every yield
  against the **clean** mid price, with no accrued interest — so book/market yields were **overstated**
  by the accrued-over-duration amount (see §2 for the real distribution; §3 for the dollar NII).
- **This folder is fixed:** §2's convention check shows `pipeline['book_yield']` now matches the
  **dirty**-price solve, and the backtest's `Y[d,i]` uses the dirty price too — removing the
  within-coupon-period **sawtooth** that could trigger phantom swaps.
- **Unchanged on purpose:** clean `price` is still the carrying/book value and the bid-ask base.
- **Next (re-run):** run this folder's `FABN_Optimizer_SAP_Backtest.ipynb` end-to-end and compare
  the cumulative net, weighted book yield, and **turnover** vs the original folder. The +10–13%
  dynamic-vs-static thesis should survive (it is a difference of two strategies sharing the same
  level shift); watch whether turnover drops once the sawtooth is gone.
